## Part 1: The Levenberg-Marquardt Interpolation

In classical Newton-Raphson optimization, we jump to the minimum by dividing the gradient by the curvature ($f''(x)$). However, this only safely converges if the function is locally convex (the curvature is positive). If we start in a non-convex region, Newton's method can violently diverge or step backwards toward a local maximum!

Gradient Descent, on the other hand, ignores curvature entirely. It safely walks downhill regardless of convexity, but it is incredibly slow when navigating flat valleys.

The **Levenberg-Marquardt algorithm** perfectly bridges these two extremes using an interpolation factor, $\gamma$:

* When **$\gamma = 1$**: The algorithm ignores the Hessian and performs pure **Gradient Descent**.
* When **$\gamma = 0$**: The algorithm performs pure **Newton-Raphson**.

**The Sandbox Task:**
Start by setting your starting position to $x_0 = -0.5$ (a non-convex "hill"). 
1. Set $\gamma = 0$ (Pure Newton) and hit play. Watch it completely fail and shoot off in the wrong direction.
2. Now, increase $\gamma$ to $0.8$ to rely heavily on Gradient Descent. Watch how it safely walks down the hill into the convex valley, and then uses the remaining Newton influence to quickly snap to the exact minimum!

In [3]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


# Define our highly non-convex objective function and its derivatives
def f(x):
    return x**4 - 4 * x**2 + x


def df(x):
    return 4 * x**3 - 8 * x + 1


def ddf(x):
    return 12 * x**2 - 8


x_vals = np.linspace(-3, 3, 200)


# Added 'alpha' to the function arguments
def interactive_lm(x_start, gamma, alpha):
    iters = 12
    history_x = [x_start]

    # Calculate the Levenberg-Marquardt path
    for _ in range(iters):
        current_x = history_x[-1]

        # Guard against dividing by exactly zero if we land on an inflection point
        second_deriv = ddf(current_x)
        if abs(second_deriv) < 1e-5:
            second_deriv = 1e-5 if second_deriv >= 0 else -1e-5

        # The 1D Levenberg-Marquardt Interpolation
        # gamma=1 -> alpha * df(x) (Gradient Descent)
        # gamma=0 -> 1/ddf(x) * df(x) (Newton Raphson)
        step_multiplier = (gamma * alpha) + ((1 - gamma) * (1.0 / second_deriv))

        next_x = current_x - step_multiplier * df(current_x)
        history_x.append(next_x)

    # Build the static background
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=f(x_vals),
            name="Objective f(x)",
            line=dict(color="lightgray", width=3),
        )
    )

    # Add the initial starting point
    fig.add_trace(
        go.Scatter(
            x=[history_x[0]],
            y=[f(history_x[0])],
            mode="markers",
            name="Start",
            marker=dict(color="black", symbol="star", size=15),
        )
    )

    # Create the Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=history_x[: k + 1],
                        y=[f(x) for x in history_x[: k + 1]],
                        mode="markers+lines",
                        name="LM Path",
                        line=dict(color="purple", width=2),
                    )
                ],
                traces=[2],
                name=f"{k}",
            )
        )
    fig.frames = frames

    # Initial empty trace for the animation to overwrite
    fig.add_trace(
        go.Scatter(
            x=[history_x[0]],
            y=[f(history_x[0])],
            mode="markers",
            marker=dict(color="purple", size=10),
            showlegend=False,
        )
    )

    # Play Controls & Slider UI (Fixed Layout)
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=300, redraw=False),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),
        xaxis=dict(range=[-3.2, 3.2]),  # Lock axes to prevent shaking
        yaxis=dict(range=[-10, 50]),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=400, redraw=False),
                                fromcurrent=True,
                                transition=dict(duration=200),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        title="Levenberg-Marquardt Optimization",
    )
    fig.show()


# 1. Start Position Slider
slider_x0 = widgets.FloatSlider(
    value=-0.5,
    min=-3.0,
    max=3.0,
    step=0.1,
    description="Start (x0):",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

# 2. Gamma Slider
slider_gamma = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Gamma (γ):",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

# 3. Alpha (Learning Rate) Slider
slider_alpha = widgets.FloatSlider(
    value=0.03,
    min=0.001,
    max=0.15,
    step=0.005,
    description="Learning Rate (α):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

# Stack the sliders vertically for a cleaner UI
ui = widgets.VBox([slider_x0, slider_gamma, slider_alpha])

# Bind all three sliders to the function
out_lm = widgets.interactive_output(
    interactive_lm, {"x_start": slider_x0, "gamma": slider_gamma, "alpha": slider_alpha}
)

# Display the UI and the plot
display(ui, out_lm)

Output()

## Part 2: Batch Learning vs. Stochastic Gradient Descent (SGD)

In standard Machine Learning (Batch Learning), our loss function $L(\vec{w})$ is the sum of the errors over the *entire* dataset. Because the dataset never changes, the loss landscape is perfectly static. If our Gradient Descent path walks into a local minimum, it will sit there forever because the gradient is exactly zero.

**The SGD Solution:**
Instead of calculating the gradient using all $M$ samples, what if we randomly pick a small "mini-batch" of $P$ samples at each step? 
Because every mini-batch contains slightly different data points, the "mini-batch gradient" will be slightly wrong (noisy) compared to the true full-dataset gradient. 

This noise is actually a superpower! It makes the algorithm bounce around erratically. If it gets stuck in a local minimum, the noisy gradient calculations will eventually "kick" it out of the trap, allowing it to find the deeper global minimum.

**The Sandbox Task:**
We have created a 2D loss landscape with a trap (a local minimum on the left) and a global minimum on the right. 
1. Set the **Batch Size to 100** (Full Batch Learning). Watch the red deterministic line permanently get stuck in the local trap.
2. Drop the **Batch Size to 10** (Mini-Batch SGD). Watch the blue dashed line bounce erratically. Notice how the noise allows it to climb out of the trap and fall into the true global minimum on the right!

In [7]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


# 1. Define our 2D Landscape: A Double-Well Trap
def f_loss(x, y):
    return (x**2 - 4) ** 2 + y**2 - 2 * x


def grad_f(x, y):
    return np.array([4 * x * (x**2 - 4) - 2, 2 * y])


# Create the static contour grid
x_grid = np.linspace(-3.5, 3.5, 100)
y_grid = np.linspace(-2.5, 2.5, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f_loss(X, Y)

# 2. Simulate a Dataset of M=100 samples with zero-centered noise
np.random.seed(42)
M_samples = 100
raw_noise = np.random.randn(M_samples, 2) * 20.0
noise_mean = np.mean(raw_noise, axis=0)
data_noise = raw_noise - noise_mean


def interactive_sgd(x_start, y_start, alpha, batch_size):
    iters = 50

    # Paths
    path_gd = [np.array([x_start, y_start])]
    path_sgd = [np.array([x_start, y_start])]

    for _ in range(iters):
        # --- BATCH LEARNING (Deterministic GD) ---
        g_gd = grad_f(path_gd[-1][0], path_gd[-1][1])
        path_gd.append(path_gd[-1] - alpha * g_gd)

        # --- STOCHASTIC GRADIENT DESCENT (Mini-batch) ---
        indices = np.random.choice(M_samples, int(batch_size), replace=False)
        batch_noise = np.mean(data_noise[indices], axis=0)
        g_sgd = grad_f(path_sgd[-1][0], path_sgd[-1][1]) + batch_noise

        path_sgd.append(path_sgd[-1] - alpha * g_sgd)

    # Convert paths for plotting
    x_gd, y_gd = zip(*path_gd)
    x_sgd, y_sgd = zip(*path_sgd)

    # 3. Build the Plot
    fig = go.Figure()

    # The Static Landscape Contour
    fig.add_trace(
        go.Contour(
            x=x_grid,
            y=y_grid,
            z=Z,
            colorscale="Greens",
            showscale=False,
            contours=dict(start=-5, end=30, size=2),
        )
    )

    # Initial Points
    fig.add_trace(
        go.Scatter(
            x=[x_gd[0]],
            y=[y_gd[0]],
            mode="markers",
            name="Full Batch GD",
            marker=dict(color="red", size=10),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[x_sgd[0]],
            y=[y_sgd[0]],
            mode="markers",
            name="Mini-Batch SGD",
            marker=dict(color="blue", symbol="x", size=10),
        )
    )

    # Create Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=x_gd[: k + 1],
                        y=y_gd[: k + 1],
                        mode="markers+lines",
                        line=dict(color="red", width=3),
                    ),
                    go.Scatter(
                        x=x_sgd[: k + 1],
                        y=y_sgd[: k + 1],
                        mode="markers+lines",
                        line=dict(color="blue", width=2, dash="dot"),
                    ),
                ],
                traces=[1, 2],
                name=f"{k}",
            )
        )
    fig.frames = frames

    # CRITICAL FIX: Changed redraw=False to redraw=True below to prevent background from vanishing
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=100, redraw=True),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),
        xaxis=dict(range=[-3.5, 3.5], title="Parameter 1"),
        yaxis=dict(range=[-2.5, 2.5], title="Parameter 2"),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=100, redraw=True),
                                fromcurrent=True,
                                transition=dict(duration=50),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=True),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        title="Escaping Local Minima with SGD Noise",
    )
    fig.show()


# UI Controls
slider_x0 = widgets.FloatSlider(
    value=-2.2, min=-3.0, max=3.0, step=0.1, description="Start X:"
)
slider_y0 = widgets.FloatSlider(
    value=0.5, min=-2.0, max=2.0, step=0.1, description="Start Y:"
)
slider_alpha = widgets.FloatSlider(
    value=0.02,
    min=0.001,
    max=0.05,
    step=0.001,
    description="Learning Rate (α):",
    style={"description_width": "initial"},
    readout_format=".3f",
)
slider_batch = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Batch Size (P):",
    style={"description_width": "initial"},
)

# Display UI
ui = widgets.VBox(
    [widgets.HBox([slider_x0, slider_y0]), widgets.HBox([slider_alpha, slider_batch])]
)
out_sgd = widgets.interactive_output(
    interactive_sgd,
    {
        "x_start": slider_x0,
        "y_start": slider_y0,
        "alpha": slider_alpha,
        "batch_size": slider_batch,
    },
)

display(ui, out_sgd)

Output()